# 6 — Multi-Agent Router Graph
## Route, Delegate, Collect — Graph-Based Specialization
⏱ ~12 min

A **supervisor LLM** reads each user message, classifies it as `PRODUCT` / `ORDER` / `SMALLTALK` / `END`, then dispatches to the right specialist sub-agent. Each specialist has its own tools and memory — they never see each other's conversation history.

By the end of this workshop you will understand *why* multi-agent graphs beat single monolithic agents, *how* `add_conditional_edges` wires a router to specialist nodes, and *how* to build both prebuilt (`create_react_agent`) and custom (`StateGraph`) specialist agents.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why multi-agent graphs? Single-agent limits |
| 2 | **Architecture** — Router pattern and graph topology |
| 3 | **Tools** — Specialist tools for product and orders domains |
| 4 | **Specialist Agents** — Prebuilt ReAct vs custom StateGraph agents |
| 5 | **Router Node** — LLM-based intent classification |
| 6 | **Full Graph** — Wiring everything with `add_conditional_edges` |
| 7 | **Demo** — End-to-end conversation walkthrough |
| 8 | **Debugging** — Inspecting routing decisions and sub-agent state |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing `requirements.txt` (or Colab)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

### Key References
> Park, J.S., O'Brien, J.C., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442  
> Wu, Q., Bansal, G., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> LangGraph multi-agent docs: https://langchain-ai.github.io/langgraph/concepts/multi_agent/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-community",
        "langchain-chroma",
        "langgraph",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import functools
import operator
import os
import uuid
from typing import Annotated, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid.\n"
    "  Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "  Colab: Secrets panel -> Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True (ends in ...{key[-4:]})")

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import create_react_agent

print("All imports OK.")

---

## Part 1 — Why Multi-Agent Graphs?

---

### The Problem with One Giant Agent

A single LLM agent given 10+ tools degrades quickly:

1. **Tool confusion** — the model picks the wrong tool when too many are available
2. **Context pollution** — orders history bleeds into product advice
3. **Prompt sprawl** — one system prompt tries to be an expert in everything
4. **Debugging pain** — you can't isolate which department got the answer wrong

### Single-Agent vs Multi-Agent

| Dimension | Single Agent | Multi-Agent Router |
|-----------|-------------|--------------------|
| **Tool selection** | Picks from all N tools | Each specialist sees only its own 2-3 tools |
| **System prompt** | One mega-prompt for all domains | Focused prompt per specialist |
| **Memory** | Shared — one conversation history | Isolated — each sub-agent has its own thread |
| **Debugging** | Hard — which step failed? | Easy — router or specialist, not both |
| **Scaling** | Add a tool → prompt gets longer | Add a specialist → rest unchanged |
| **Cost** | High — all context every call | Lower — only relevant specialist called |

### Routing Strategies Compared

| Strategy | How routing works | Best for |
|----------|-------------------|----------|
| **LLM keyword router** (this example) | LLM returns one word: `PRODUCT`, `ORDER`, etc. | Intent classification across distinct domains |
| **Rule-based router** | `if 'order' in message.lower()` | Simple, predictable categories with clear keywords |
| **Embedding similarity** | cosine sim to category exemplars | Ambiguous natural language, many categories |
| **Tool-calling router** | LLM calls `route(destination=...)` tool | When you need structured reasoning about the route |
| **Shared-state handoff** | Agents add to a common state dict | Collaborative workflows where agents build on each other |

This example uses the **LLM keyword router** — simple, explainable, and easy to extend.

---

## Part 2 — Architecture: The Router Graph

---

### How `add_conditional_edges` Works

LangGraph's `add_conditional_edges` is the key mechanism. It takes:
1. A **source node** (the node that just ran)
2. A **routing function** (reads state and returns a string key)
3. A **mapping dict** (string key to destination node name)

```python
graph.add_conditional_edges(
    "Router",          # after Router runs...
    find_route,        # ...call find_route(state) which returns a string...
    {
        "PRODUCT":   "Product_Agent",   # ...and map that string to a node
        "ORDER":     "Orders_Agent",
        "SMALLTALK": "Small_Talk",
        "END":       END,
    }
)
```

### Graph Topology

```
+----------------------------------------------------------+
|                   RouterAgentState                        |
|           messages: Annotated[list, operator.add]         |
+----------------------------------------------------------+
                           |
                           v
                    +-------------+
                    |    START    |
                    +------+------+
                           |
                           v
                    +-------------+
                    |   Router    |  <- LLM: read user intent
                    |   (LLM)     |    output: PRODUCT / ORDER /
                    +------+------+           SMALLTALK / END
                           |
          add_conditional_edges(find_route, {...})
                           |
          +----------------+-----------------+
          |                |                 |
          v                v                 v
  +--------------+  +--------------+  +--------------+
  | Product_Agent|  | Orders_Agent |  |  Small_Talk  |
  |              |  |              |  |    (LLM)     |
  | create_react |  | custom       |  |              |
  | _agent()     |  | StateGraph   |  | No tools     |
  |              |  |              |  |              |
  | Tools:       |  | Tools:       |  |              |
  | get_features |  | get_order    |  |              |
  | get_price    |  | update_qty   |  |              |
  +------+-------+  +------+-------+  +------+-------+
         |                 |                 |
         +-----------------+-----------------+
                           |
                           v
                    +-------------+
                    |     END     |
                    +-------------+
```

### Key Design Decisions

| Decision | Choice | Why |
|----------|--------|-----|
| **State accumulation** | `operator.add` on messages list | All turns visible for debugging |
| **Sub-agent memory** | Separate `MemorySaver` per agent | Prevents cross-domain context leak |
| **Thread ID** | UUID per session, passed as config | Isolates conversation history |
| **Router output** | Single keyword string | Deterministic, easy to unit-test |
| **One-way routing** | Specialists return to END, not back to Router | Simpler flow; each query is one round-trip |

---

## Part 3 — Specialist Tools

---

The original `src/tools.py` loads CSV files from disk. This notebook uses inline dicts — same tool interfaces, no file I/O required.

### Tool Design Principles

| Principle | Example |
|-----------|--------|
| **One responsibility** | `get_product_features` never touches orders data |
| **Clear docstrings** | The LLM reads the docstring to decide when to call the tool |
| **Predictable return types** | Always return a `str` — the LLM context is text-only |
| **Graceful not-found** | Return a message string, not an exception |
| **No side effects in read tools** | `get_order_details` never modifies state |

In [ ]:
# ─── 3-A: Inline product catalog and order data ───────────────────────────────
# In production (src/tools.py), these come from CSV/PDF files.
# Here we use dicts to keep the notebook self-contained.

PRODUCT_CATALOG = {
    "SpectraBook": {
        "cpu": "Intel Core Ultra 7",
        "ram": "16GB DDR5",
        "storage": "512GB NVMe SSD",
        "display": "14-inch IPS 2560x1600",
        "battery": "72Wh, up to 12h",
        "price": 1299,
    },
    "AlphaBook Pro": {
        "cpu": "AMD Ryzen 9 7940H",
        "ram": "32GB DDR5",
        "storage": "1TB NVMe SSD",
        "display": "15.6-inch OLED 2880x1800",
        "battery": "86Wh, up to 10h",
        "price": 1799,
    },
    "NovaPad": {
        "cpu": "Apple M3",
        "ram": "8GB unified memory",
        "storage": "256GB SSD",
        "display": "13.3-inch Retina",
        "battery": "58Wh, up to 18h",
        "price": 999,
    },
}

ORDERS = {
    "ORD-7311": {"product": "SpectraBook",   "quantity": 2, "delivery": "2025-07-15"},
    "ORD-8842": {"product": "AlphaBook Pro", "quantity": 1, "delivery": "2025-07-20"},
    "ORD-6948": {"product": "NovaPad",       "quantity": 3, "delivery": "2025-07-28"},
}

print(f"Catalog: {list(PRODUCT_CATALOG.keys())}")
print(f"Orders:  {list(ORDERS.keys())}")

In [ ]:
# ─── 3-B: Product tools ───────────────────────────────────────────────────────
# These are assigned ONLY to the Product_Agent — the Orders_Agent never sees them.

# @tool makes the function callable by the LLM; the docstring IS the tool description
# — keep docstrings precise: vague descriptions cause the LLM to pick the wrong tool
@tool
def get_product_features(product_name: str) -> str:
    """Returns detailed technical specifications and features for a laptop given its name.
    Performs a case-insensitive substring match. Returns an error message if no match found."""
    for key, val in PRODUCT_CATALOG.items():
        if product_name.lower() in key.lower():
            specs = ", ".join(f"{k}: {v}" for k, v in val.items() if k != "price")
            return f"{key} specifications -- {specs}"
    return f"Product '{product_name}' not found. Available: {', '.join(PRODUCT_CATALOG.keys())}"


@tool
def get_laptop_price(product_name: str) -> str:
    """Returns the retail price of a laptop given its name.
    Performs a case-insensitive substring match. Returns an error message if no match found."""
    for key, val in PRODUCT_CATALOG.items():
        if product_name.lower() in key.lower():
            return f"{key} is priced at USD {val['price']:,}"
    return f"Product '{product_name}' not found. Available: {', '.join(PRODUCT_CATALOG.keys())}"


# Quick smoke tests — verify tools work before wiring into agents
print(get_product_features.invoke({"product_name": "spectra"}))
print(get_laptop_price.invoke({"product_name": "alpha"}))
print(get_laptop_price.invoke({"product_name": "unknown"}))

In [ ]:
# ─── 3-C: Orders tools ────────────────────────────────────────────────────────
# These are assigned ONLY to the Orders_Agent — the Product_Agent never sees them.

@tool
def get_order_details(order_id: str) -> str:
    """Returns details about a laptop order given its order ID (e.g. ORD-7311).
    Returns product name, quantity ordered, and estimated delivery date.
    Returns an error message if the order ID does not exist."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"Order '{order_id}' not found. Available orders: {', '.join(ORDERS.keys())}"
    return (
        f"Order {order_id}: {order['product']}, "
        f"quantity {order['quantity']}, "
        f"estimated delivery {order['delivery']}"
    )


# update_quantity mutates in-memory ORDERS dict — in production this would write to a DB
# The docstring signals to the LLM when to use this tool vs get_order_details
@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """Updates the quantity for a laptop order given its order ID and the new quantity.
    Returns a confirmation message or an error if the order is not found."""
    if order_id.upper() not in ORDERS:
        return f"Order '{order_id}' not found."
    old_qty = ORDERS[order_id.upper()]["quantity"]
    ORDERS[order_id.upper()]["quantity"] = new_quantity
    return f"Updated order {order_id}: quantity changed from {old_qty} to {new_quantity}."


# Smoke tests
print(get_order_details.invoke({"order_id": "ORD-7311"}))
print(update_quantity.invoke({"order_id": "ORD-7311", "new_quantity": 5}))
print(get_order_details.invoke({"order_id": "ORD-7311"}))  # verify update
ORDERS["ORD-7311"]["quantity"] = 2  # reset for the rest of the notebook

---

## Part 4 — Specialist Agents

---

This example demonstrates **two different ways** to build a specialist agent in LangGraph:

### Option A: `create_react_agent` (Prebuilt)

```
  Input messages
       |
       v
  +----------+
  |   LLM   |  <- decides: answer directly or call a tool
  +----+-----+
       |
  tool_call?  YES --------> +----------+
       |                    |  Tools   |
       |                    +----+-----+
       |                         |
       |  <------ ToolMessage ---+
       |  (loop back to LLM)
       |
  NO --+-> Return final answer
```

### Option B: Custom `StateGraph` Agent

```
  Input messages
       |
       v
  +--------------+
  |  orders_llm  |  node: call LLM with bound tools
  +------+-------+
         |
  is_tool_call?  TRUE -----> +--------------+
         |                   | orders_tools |  node: execute each tool_call
         |                   +------+-------+  return ToolMessage results
         |                          |
         |  <------- ToolMessages --+
         |  (back to orders_llm)
         |
  FALSE -+-> END
```

### Prebuilt vs Custom Comparison

| | `create_react_agent` | Custom `StateGraph` |
|--|---------------------|---------------------|
| **Lines of code** | ~5 | ~50 |
| **Loop control** | Automatic | Manual (`is_tool_call` conditional) |
| **Tool execution** | Automatic | Manual (`call_tools` node) |
| **Customization** | Limited to params | Full control over every node |
| **Error handling** | Default | Custom fallback logic per tool |
| **When to use** | 90% of use cases | Custom retry, human-in-loop, non-standard flows |

In [ ]:
# ─── 4-A: Product Agent — prebuilt create_react_agent ─────────────────────────
# create_react_agent handles the LLM -> tool call -> LLM loop automatically.
# We only need to pass the model, tools, system prompt, and a checkpointer.

# temperature=0 for deterministic routing — higher values cause inconsistent PRODUCT/ORDER labels
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# create_react_agent: Reason + Act loop — LLM decides tool, tool runs, result fed back to LLM
# Alternative: build a custom StateGraph (see cell 4-B) for finer control over each step
product_agent = create_react_agent(
    model=llm,
    tools=[get_product_features, get_laptop_price],
    prompt=SystemMessage(
        "You are a professional laptop product specialist. "
        "Answer questions about laptop specifications and pricing using ONLY the available tools. "
        "Be concise and factual. Do not guess specs from memory."
    ),
    checkpointer=MemorySaver(),  # enables multi-turn memory within the product domain
)

# Test in isolation before wiring into the router graph
test_config = {"configurable": {"thread_id": "product-test-1"}}
test_result = product_agent.invoke(
    {"messages": [HumanMessage("What are the specs of the SpectraBook?")]},
    test_config
)
print("Product Agent isolated test:")
print(test_result["messages"][-1].content)

In [ ]:
# ─── 4-B: Orders Agent — custom StateGraph ────────────────────────────────────
# This mirrors src/agent/orders.py from the production implementation.
# The custom graph gives full control over tool dispatch and error handling.
#
# NOTE: The production orders.py has a subtle bug — the return statement is
# inside the for loop in call_tools, so only the first tool call is processed.
# This version fixes that. See Exercise 4 in the answer key for details.


# TypedDict gives LangGraph typed access to state keys without runtime overhead
# Annotated[list, operator.add] tells LangGraph to append messages rather than overwrite
class OrdersAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class OrdersAgent:
    """Custom ReAct-style agent for order management.

    Manually implements the LLM -> tool_call -> tool_result -> LLM loop
    using a StateGraph instead of the prebuilt create_react_agent.
    This gives full control over tool dispatch and error handling.
    """

    def __init__(self, model, tools, system_prompt: str):
        self.system_prompt = system_prompt
        # Bind tools to the model so it knows their schemas during generation
        self.model = model.bind_tools(tools)
        # Build a lookup dict for fast tool dispatch by name
        self.tools = {t.name: t for t in tools}

        graph = StateGraph(OrdersAgentState)
        graph.add_node("orders_llm", self.call_llm)
        graph.add_node("orders_tools", self.call_tools)
        # Conditional: if the LLM requested a tool call, run it; else stop
        graph.add_conditional_edges(
            "orders_llm",
            self.is_tool_call,
            {True: "orders_tools", False: END}
        )
        # After tools run, always go back to the LLM to formulate a response
        graph.add_edge("orders_tools", "orders_llm")
        graph.set_entry_point("orders_llm")
# compile() locks the graph topology — no nodes or edges can be added after this
# MemorySaver() here scopes memory to this sub-agent; swappable with SqliteSaver for persistence
        self.agent_graph = graph.compile(checkpointer=MemorySaver())

    def call_llm(self, state: OrdersAgentState) -> dict:
        messages = [SystemMessage(self.system_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        return {"messages": [result]}

    def is_tool_call(self, state: OrdersAgentState) -> bool:
        last = state["messages"][-1]
        return hasattr(last, "tool_calls") and len(last.tool_calls) > 0

    def call_tools(self, state: OrdersAgentState) -> dict:
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for tc in tool_calls:
            if tc["name"] not in self.tools:
                content = f"Unknown tool: {tc['name']}"
            else:
                content = str(self.tools[tc["name"]].invoke(tc["args"]))
            results.append(ToolMessage(
                tool_call_id=tc["id"],
                name=tc["name"],
                content=content
            ))
        # IMPORTANT: return is OUTSIDE the for loop so all tool calls are processed
        return {"messages": results}


orders_agent_obj = OrdersAgent(
    model=llm,
    tools=[get_order_details, update_quantity],
    system_prompt=(
        "You are a professional order management assistant. "
        "Retrieve and update laptop orders using ONLY the available tools. "
        "Do NOT reveal details of orders other than the one requested."
    ),
)

# Test in isolation
test_config2 = {"configurable": {"thread_id": "orders-test-1"}}
test_result2 = orders_agent_obj.agent_graph.invoke(
    {"messages": [HumanMessage("What is the status of order ORD-8842?")]},
    test_config2
)
print("Orders Agent isolated test:")
print(test_result2["messages"][-1].content)

---

## Part 5 — Router Node: LLM-Based Intent Classification

---

The router uses an LLM call with a tightly constrained system prompt. The key design rule: **the prompt must instruct the model to return exactly one word**. Any ambiguity in the instruction leads to multi-word responses that break the routing dict lookup.

### Router Prompt Engineering

```
Bad:  "Classify the message as one of: PRODUCT, ORDER, SMALLTALK, or END"
         -> Model may return "The message is PRODUCT" (breaks dict lookup)

Good: "Your output MUST be exactly one word from: PRODUCT, ORDER, SMALLTALK, or END."
         -> Model learns to return exactly "PRODUCT"
```

The `find_route` function reads the last message content (the router's single-word response) and returns it as the routing key. A `.strip().upper()` call guards against whitespace and case variations.

In [ ]:
# ─── 5-A: Router system prompt and classification demo ────────────────────────

# Constrain output to a single token — makes routing deterministic and easy to map to graph edges
# Alternative: with_structured_output(RouterEnum) — but adds latency and costs extra tokens
ROUTER_SYSTEM_PROMPT = """
You are a Router. Analyze the input query and choose exactly one of these 4 options:

PRODUCT   -- The query is about laptop product specifications, features, comparisons, or pricing.
ORDER     -- The query is about orders: status, details, quantity updates, or delivery dates.
SMALLTALK -- The query is small talk, a greeting, a goodbye, or general conversation.
END       -- Default for anything that does not fit the above categories.

Your output MUST be exactly one word from: PRODUCT, ORDER, SMALLTALK, END.
Do not include any other text, punctuation, or explanation.
""".strip()

# Test the router classification in isolation before wiring into the graph
test_queries = [
    "Tell me about the SpectraBook features",
    "What is the status of order ORD-7311?",
    "Hi there, how are you doing today?",
    "Can you update the quantity of ORD-8842 to 3?",
    "What is the capital of France?",   # out of domain -> END
    "How much does the AlphaBook Pro cost?",
]

print("Router classification test:")
print(f"{'Query':<50} Route")
print("-" * 65)
for q in test_queries:
    messages = [SystemMessage(ROUTER_SYSTEM_PROMPT), HumanMessage(q)]
    result = llm.invoke(messages)
    route = result.content.strip().upper()
    print(f"{q:<50} {route}")

In [ ]:
# ─── 5-B: Small talk handler ──────────────────────────────────────────────────
# The Small_Talk node is a lightweight LLM call — no tools needed.
# It uses its own focused system prompt to stay on-topic.

SMALLTALK_PROMPT = """
You are a professional customer support assistant for a laptop company.
The user is making small talk. Respond warmly and professionally.
Mention that you can help with laptop product questions and order management.
Keep your response concise (2-3 sentences).
""".strip()

# Test in isolation
st_result = llm.invoke([
    SystemMessage(SMALLTALK_PROMPT),
    HumanMessage("Hi! How are you doing today?")
])
print("Small talk test:")
print(st_result.content)

---

## Part 6 — Wiring the Full Router Graph

---

Now we connect all the pieces: router node, specialist nodes, conditional edges, and graph compilation.

### `functools.partial` for Node Functions

LangGraph node functions must have the signature `(state) -> dict`. But our specialist agents need to be passed as arguments. `functools.partial` binds the agent argument ahead of time:

```python
# Without partial -- wrong signature, LangGraph can't call this
def agent_node(state, agent):   # two args
    ...

# With partial -- binds agent, leaving only state
product_node = functools.partial(agent_node, agent=product_agent)  # one arg
graph.add_node("Product_Agent", product_node)  # works
```

### Why `operator.add` on the Messages List?

```python
class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
```

Without `operator.add`, each node would **replace** the messages list when it returns `{"messages": [new_msg]}`. With `operator.add`, the new message is **appended** to the existing list. This means the state accumulates the full conversation trace across all nodes — invaluable for debugging.

In [ ]:
# ─── 6-A: State schema and helper functions ───────────────────────────────────

class RouterAgentState(TypedDict):
    # operator.add means messages are accumulated (appended), not replaced.
    # Every node returns {"messages": [new_msg]} and the list grows.
    messages: Annotated[list[AnyMessage], operator.add]


def agent_node(state: RouterAgentState, agent) -> dict:
    """Invoke a sub-agent with the current state and wrap its output as an AIMessage.

    Each sub-agent gets a fresh thread_id so its MemorySaver history is
    isolated from other sub-agents -- product and order memories never mix.
    """
# Fresh uuid per call so each sub-agent invocation gets an isolated MemorySaver thread
# To share memory across turns within a domain, pass a stable session-scoped thread_id instead
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = agent.invoke(state, config)
    return {"messages": [AIMessage(content=result["messages"][-1].content)]}


def router_llm_node(state: RouterAgentState) -> dict:
    """The Router node: classifies user intent into PRODUCT/ORDER/SMALLTALK/END."""
    messages = [SystemMessage(ROUTER_SYSTEM_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    route = result.content.strip().upper()
    print(f"  [router -> {route}]")
    return {"messages": [AIMessage(content=route)]}


# add_conditional_edges maps find_route's return value to a node name in the routing dict
# The string returned here MUST exactly match a key in the dict passed to add_conditional_edges
def find_route(state: RouterAgentState) -> str:
    """Routing function: reads the router's output and returns the destination key.

    Called by add_conditional_edges after the Router node runs.
    The returned string must exactly match a key in the routing dict.
    """
    return state["messages"][-1].content.strip().upper()


def smalltalk_node(state: RouterAgentState) -> dict:
    """Small Talk node: responds to greetings and general chat."""
    messages = [SystemMessage(SMALLTALK_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    return {"messages": [result]}


# Bind specialist agents into node-compatible single-argument functions
product_node = functools.partial(agent_node, agent=product_agent)
orders_node  = functools.partial(agent_node, agent=orders_agent_obj.agent_graph)

print("Node functions ready.")

In [ ]:
# ─── 6-B: Build and compile the router graph ──────────────────────────────────

router_graph = StateGraph(RouterAgentState)

# Register all four nodes
router_graph.add_node("Router",        router_llm_node)
router_graph.add_node("Product_Agent", product_node)
router_graph.add_node("Orders_Agent",  orders_node)
router_graph.add_node("Small_Talk",    smalltalk_node)

# The core routing mechanism:
# After Router runs, call find_route(state) and map its return value to a node.
router_graph.add_conditional_edges(
    "Router",
    find_route,
    {
        "PRODUCT":   "Product_Agent",
        "ORDER":     "Orders_Agent",
        "SMALLTALK": "Small_Talk",
        "END":       END,
    }
)

# One-way routing: specialists return directly to END (no back-to-router loop).
# Each user message is handled in a single router -> specialist -> END round-trip.
router_graph.add_edge("Product_Agent", END)
router_graph.add_edge("Orders_Agent",  END)
router_graph.add_edge("Small_Talk",    END)

# Every invocation starts at the Router node
router_graph.set_entry_point("Router")

# No checkpointer on the outer router — each invoke() is stateless at the router level
# Sub-agents carry their own MemorySaver; the router only needs to route, not remember
router_app = router_graph.compile()

print("Router graph compiled.")
print("Nodes:", list(router_graph.nodes.keys()))

---

## Part 7 — Demo: End-to-End Conversation

---

We now run a realistic multi-turn conversation. Notice:
- Each message travels through the **Router** first
- The `[router -> X]` print shows which specialist was chosen
- Product and order memory are fully isolated from each other

In [ ]:
# ─── 7-A: Single-message routing demo ────────────────────────────────────────
# Each call is independent — a fresh message in, a fresh response out.

demo_config = {"configurable": {"thread_id": "demo-session-1"}}

single_queries = [
    "Hi there! What can you help me with?",
    "Tell me about the SpectraBook features",
    "What is the status of order ORD-7311?",
    "How much does the AlphaBook Pro cost?",
    "Please update order ORD-7311 to 4 units",
]

for query in single_queries:
    print(f"\nUSER: {query}")
    result = router_app.invoke(
        {"messages": [HumanMessage(content=query)]},
        demo_config
    )
    print(f"AGENT: {result['messages'][-1].content}")

In [ ]:
# ─── 7-B: Full multi-turn conversation ────────────────────────────────────────
# A realistic support session with mixed intents across all four routes.

conversation = [
    "How are you doing?",
    "Show me the details of order ORD-7311",
    "Can you add one more laptop to that order?",
    "Tell me about the features of the SpectraBook",
    "How much does it cost?",
    "What programming language is it built in?",  # out of scope -> END (silent)
    "Bye!",
]

session_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("=" * 60)
print("MULTI-TURN SUPPORT CONVERSATION")
print("=" * 60)

for turn, user_input in enumerate(conversation, 1):
    print(f"\n[Turn {turn}]")
    print(f"USER:  {user_input}")
    response = router_app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        session_config
    )
    last_msg = response["messages"][-1]
    print(f"AGENT: {last_msg.content}")

---

## Part 8 — Debugging: Inspecting Routing Decisions

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Misrouted query** | Product question hits Orders_Agent | Router prompt ambiguous | Tighten examples in ROUTER_SYSTEM_PROMPT |
| **Route key mismatch** | `KeyError` in conditional edges | Router returned multi-word string | Add `.strip().upper()` in `find_route` |
| **Memory leak across domains** | Order context bleeds into product answers | Shared thread_id for sub-agents | Use `uuid.uuid4()` per sub-agent call |
| **Tool not found** | `Unknown tool` error | Tool name mismatch in `self.tools` dict | Check `@tool` function names match dict keys |
| **Silent END** | No AGENT response for out-of-scope queries | `END` route exits graph immediately | Add an out-of-scope fallback node (see Exercise 3) |
| **Parallel tool calls dropped** | Only first tool result used | `return` inside tool loop | Move `return` outside the `for` loop |

In [ ]:
# ─── 8-A: Inspect full message state after a query ────────────────────────────
# The RouterAgentState accumulates ALL messages via operator.add.
# Printing them shows exactly which nodes ran and what each returned.

debug_result = router_app.invoke(
    {"messages": [HumanMessage("What is the delivery date for ORD-6948?")]},
    {"configurable": {"thread_id": "debug-session"}}
)

print("Full message trace (all nodes' outputs accumulated):")
print("-" * 55)
for i, msg in enumerate(debug_result["messages"]):
    msg_type = type(msg).__name__
    content_preview = msg.content[:120].replace("\n", " ")
    print(f"  [{i}] {msg_type:<15}: {content_preview}")

In [ ]:
# ─── 8-B: Unit test the router in isolation ───────────────────────────────────
# Before debugging the full graph, verify the router classification alone.
# This is the fastest way to catch prompt engineering issues.

def classify(query: str) -> str:
    """Run just the router LLM and return the route string."""
    messages = [SystemMessage(ROUTER_SYSTEM_PROMPT), HumanMessage(query)]
    result = llm.invoke(messages)
    return result.content.strip().upper()


# Edge cases likely to trip up the router
edge_cases = [
    ("Compare SpectraBook vs AlphaBook Pro",  "PRODUCT"),    # product comparison
    ("Cancel my order ORD-7311",              "ORDER"),      # unsupported action but order-related
    ("What's the weather like today?",        "END"),        # completely out of domain
    ("I need help",                           "SMALLTALK"),  # ambiguous
    ("SpectraBook order quantity update",      "ORDER"),      # mixed keywords
]

print(f"{'Query':<45} {'Expected':<12} {'Got':<12} {'Pass?'}")
print("-" * 80)
for query, expected in edge_cases:
    got = classify(query)
    passed = "PASS" if got == expected else "FAIL"
    print(f"{query:<45} {expected:<12} {got:<12} {passed}")

---

## Exercises

---

### Exercise 1 — Add a COMPARE Route

Add a new `COMPARE` route to the router that handles queries like "compare SpectraBook vs AlphaBook Pro". Requirements:
- Add `COMPARE` as a fifth keyword in `ROUTER_SYSTEM_PROMPT`
- Create a `compare_products` tool that accepts two product names and returns a side-by-side spec table
- Create a `Compare_Agent` node using `create_react_agent` with that tool
- Wire it into `add_conditional_edges`

**Starter below:**

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

@tool
def compare_products(product_a: str, product_b: str) -> str:
    """Compares two laptops side by side given their names."""
    # TODO: look up both products in PRODUCT_CATALOG
    # TODO: return a formatted side-by-side comparison string
    return "TODO: implement comparison"


# TODO: create compare_agent using create_react_agent
# compare_agent = create_react_agent(
#     model=llm,
#     tools=[compare_products],
#     prompt=SystemMessage("You are a product comparison specialist..."),
#     checkpointer=MemorySaver(),
# )

# TODO: update ROUTER_SYSTEM_PROMPT to include COMPARE as a fifth option
# TODO: add compare_node = functools.partial(agent_node, agent=compare_agent)
# TODO: rebuild router_graph with the new node and edge

print("Exercise 1: implement compare_products, then rebuild the graph.")

### Exercise 2 — Persistent Sub-Agent Memory

Currently each sub-agent call gets a fresh `uuid.uuid4()` thread_id in `agent_node`, so no specialist ever remembers previous turns with that domain.

**Goal:** Make follow-up questions work. After "Tell me about SpectraBook", the next message "How much does it cost?" should remember the product context.

**Hint:** Thread the outer session's thread_id into the sub-agent config, namespaced by domain (e.g., `"{session_id}::product"`). You will need to extend `RouterAgentState` to carry a `session_id` field, or pass it through `functools.partial`.

**Challenge:** How do you prevent the orders memory from leaking into the product agent when both are used in the same session?

### Exercise 3 — Graceful END Handler

When the router returns `END`, the graph terminates silently — the user gets no response. Add a graceful fallback:

1. Create an `out_of_scope_node` that returns a polite "I can only help with product questions and orders" message
2. Map `"END"` in `add_conditional_edges` to this node instead of the bare `END` constant
3. Add an edge from `out_of_scope_node` to `END`

### Exercise 4 — Bug Hunt in the Custom Agent

Read `src/agent/orders.py` (the original production file). Find the bug in `call_tools` that causes only the first tool call to be executed when the LLM requests multiple tool calls in one turn.

**Questions:**
- Where exactly is the bug?
- What query would trigger it? (Hint: ask for the price AND delivery date of an order at the same time)
- Fix it with a one-line change
- When would you choose the custom `StateGraph` over `create_react_agent`? Name two concrete scenarios.

---

## What's Next?

You now understand the supervisor/router pattern — the foundation of most production multi-agent systems. Here's where to go next:

### Related Examples in This Repo

| Example | What it adds | Path |
|---------|-------------|------|
| **5-react-agent-lg** | Two-agent critic loop without a router | `../5-react-agent-lg/` |
| **19-multi-agent-notebook** | Multi-agent collaboration with shared state | `../19-multi-agent-notebook/` |
| **25-adaptive-rag** | Same routing idea applied to retrieval strategies | `../25-adaptive-rag/` |
| **30-agentic-rag** | Full agentic RAG with tool-using agents | `../30-agentic-rag/` |

### Patterns to Explore Next

- **Hierarchical multi-agent**: Supervisors delegating to supervisors (tree of routers)
- **Agent handoff**: An agent passes partial results to the next via `Command` (LangGraph 0.2+)
- **Shared memory**: Agents writing to a common state store for collaborative tasks
- **LangGraph Cloud**: Deploy this graph as a stateful REST API with built-in persistence

### Further Reading

- Park et al. (2023). *Generative Agents.* https://arxiv.org/abs/2304.03442
- Wu et al. (2023). *AutoGen: Multi-Agent Conversation Framework.* https://arxiv.org/abs/2308.08155
- LangGraph multi-agent patterns: https://langchain-ai.github.io/langgraph/concepts/multi_agent/
- LangGraph supervisor tutorial: https://langchain-ai.github.io/langgraph/tutorials/multi_agent/agent_supervisor/

---

*Workshop complete. The natural next step is example 19 — multi-agent collaboration with shared state.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 Answer — COMPARE Route

**Key insight:** Adding a new route requires changes in exactly three places — the router system prompt, the tool/agent definition, and the `add_conditional_edges` dict. Nothing else in the graph changes. This is the extension-point value of the router pattern.

In [ ]:
# Answer Key — Exercise 1: COMPARE route

@tool
def compare_products_answer(product_a: str, product_b: str) -> str:
    """Compares two laptops side by side given their names."""
    results = {}
    for name in [product_a, product_b]:
        for key, val in PRODUCT_CATALOG.items():
            if name.lower() in key.lower():
                results[key] = val
                break
    if len(results) < 2:
        return f"Could not find both products. Available: {', '.join(PRODUCT_CATALOG.keys())}"

    names = list(results.keys())
    lines = [f"{'Spec':<20} {names[0]:<25} {names[1]:<25}", "-" * 70]
    for spec in ["cpu", "ram", "storage", "display", "battery", "price"]:
        a_val = str(results[names[0]].get(spec, "N/A"))
        b_val = str(results[names[1]].get(spec, "N/A"))
        lines.append(f"{spec:<20} {a_val:<25} {b_val:<25}")
    return "\n".join(lines)


# Test the comparison tool
print(compare_products_answer.invoke({"product_a": "SpectraBook", "product_b": "AlphaBook"}))

# The updated ROUTER_SYSTEM_PROMPT would add one line:
#   COMPARE   -- The query asks to compare two or more laptop models side by side.
#
# The updated conditional edges dict would add one entry:
#   "COMPARE": "Compare_Agent"

### Exercise 2 Answer — Persistent Sub-Agent Memory

**Key insight:** The outer router graph does not have a MemorySaver (we did not compile it with `checkpointer=`), so the session thread_id lives only in the `config` dict passed to `router_app.invoke`. We need to thread that ID down into each sub-agent call — but namespace it per domain to prevent cross-contamination.

The cleanest approach extends `RouterAgentState` with a `session_id` field, set once by the caller and propagated automatically through all nodes via LangGraph's state merging.

In [ ]:
# Answer Key — Exercise 2: persistent domain-scoped memory

class RouterAgentStateWithSession(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    session_id: str  # set once by the caller; propagated through all nodes


def agent_node_persistent(state: RouterAgentStateWithSession, agent, domain: str) -> dict:
    """Uses a domain-scoped thread_id so specialist memories persist across turns
    within a session, but remain isolated between domains."""
    session_id = state.get("session_id", str(uuid.uuid4()))
    # e.g. 'abc123::product' or 'abc123::orders'
    thread_id = f"{session_id}::{domain}"
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(state, config)
    return {"messages": [AIMessage(content=result["messages"][-1].content)]}


# Usage pattern:
# product_node_p = functools.partial(agent_node_persistent, agent=product_agent, domain="product")
# orders_node_p  = functools.partial(agent_node_persistent, agent=orders_agent_obj.agent_graph, domain="orders")
# Then rebuild the graph with these nodes.
#
# Invoke with:
# my_session = str(uuid.uuid4())
# router_app_p.invoke({"messages": [...], "session_id": my_session}, config)

print("With session_id='abc123':")
print("  Product thread: abc123::product")
print("  Orders thread:  abc123::orders")
print("These are isolated -- product questions never see order history.")

### Exercise 3 Answer — Graceful END Handler

**Key insight:** Never let `END` be the direct destination of a user-facing route. Always provide a node that generates a response first — the user should always receive an acknowledgment, even for out-of-scope queries.

In [ ]:
# Answer Key — Exercise 3: graceful out-of-scope handler

def out_of_scope_node(state: RouterAgentState) -> dict:
    """Returns a polite fallback when the query is outside the agent's scope."""
    response = (
        "I'm sorry, I can only help with laptop product questions "
        "(specifications and pricing) and order management "
        "(status, details, and quantity updates). "
        "Please ask about one of those topics."
    )
    return {"messages": [AIMessage(content=response)]}


# In the graph definition, make these two changes:
#
# Change in add_conditional_edges:
#   "END": END
#   becomes:
#   "END": "Out_Of_Scope"
#
# Add these two lines:
#   router_graph.add_node("Out_Of_Scope", out_of_scope_node)
#   router_graph.add_edge("Out_Of_Scope", END)

# Quick test of the node in isolation
state = {"messages": [HumanMessage("What is the capital of France?")]}
result = out_of_scope_node(state)
print("Out-of-scope response:")
print(result["messages"][0].content)

### Exercise 4 Answer — Bug Hunt in the Custom Agent

**The bug:** In `src/agent/orders.py`, the `return` statement in `call_tools` is **inside** the `for tc in tool_calls` loop (line 99). This means only the **first** tool call is executed — if the LLM requests two tools in one turn (parallel tool calls), the second is silently dropped.

```python
# BUGGY (production src/agent/orders.py lines 77-99)
for tool in tool_calls:
    result = self.tools[tool["name"]].invoke(tool["args"])
    results.append(ToolMessage(tool_call_id=tool["id"], ...))
    if self.debug:
        print(f"\nTools returned {results}")
    return {"messages": results}     # <- BUG: inside the loop, exits after first tool

# FIXED (one-line change: dedent the return)
for tool in tool_calls:
    result = self.tools[tool["name"]].invoke(tool["args"])
    results.append(ToolMessage(tool_call_id=tool["id"], ...))
    if self.debug:
        print(f"\nTools returned {results}")
return {"messages": results}         # <- FIXED: outside the loop
```

**Query that triggers it:** Ask the agent to retrieve order details AND update the quantity in a single message, e.g., "Get me the details of ORD-7311 and change the quantity to 5." The LLM will issue two tool_calls simultaneously; only `get_order_details` would run with the bug.

**When to choose custom `StateGraph` over `create_react_agent`:**
1. **Custom retry logic**: If a tool returns an error, retry up to 3 times with exponential backoff before surfacing the failure — impossible with the prebuilt.
2. **Human-in-the-loop approval**: Insert a `wait_for_approval` node before any destructive operation (e.g., `update_quantity`) that pauses execution until a human confirms — the custom graph makes this a natural node in the flow.